# 第4章 演習：Human-in-the-Loop (HITL) の実装

## この演習について

`# TODO:` のある箇所にコードを記述して、重要な操作の前に人間の承認を求める
HITL エージェントを完成させてください。

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap04-exercise"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("環境変数の設定が完了しました")

## 準備: ツールの定義（提供済み）

以下のダミーツールはあらかじめ定義してあります。そのまま使用してください。

In [ ]:
from langchain.tools import tool

# 提供済みツール（変更不要）
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """指定したアドレスにメールを送信します。この操作は外部システムに影響を与えます。
    
    Args:
        to: 送信先のメールアドレス
        subject: メールの件名
        body: メールの本文
    """
    print(f"[メール送信] 宛先: {to}, 件名: {subject}")
    return f"メールを送信しました: {to} 宛, 件名: '{subject}'"


@tool
def delete_record(record_id: str, table: str) -> str:
    """指定したテーブルからレコードを削除します。この操作は取り消せません。
    
    Args:
        record_id: 削除するレコードの ID
        table: 対象のテーブル名
    """
    print(f"[データ削除] テーブル: {table}, レコードID: {record_id}")
    return f"レコード {record_id} を {table} テーブルから削除しました"


@tool
def get_user_info(user_id: str) -> str:
    """ユーザー情報を取得します（読み取り専用）。
    
    Args:
        user_id: 情報を取得するユーザーの ID
    """
    return f"ユーザー情報: ID={user_id}, 名前=山田太郎, メール=yamada@example.com"

print("ツールの準備が完了しました")

## 課題1: HITL エージェントを作成する

**目標:** `HumanInTheLoopMiddleware` を使って、`send_email` と `delete_record` ツールの
実行前に必ず承認を求めるエージェントを作成してください。

**ヒント:**
- `from langchain.middleware import HumanInTheLoopMiddleware` でインポート
- `HumanInTheLoopMiddleware(interrupt_on={"ツール名1", "ツール名2"})` で指定
- `create_agent` に `middleware=[hitl_middleware]` と `checkpointer=InMemorySaver()` を追加

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# TODO: HumanInTheLoopMiddleware をインポートしてください
# from ??? import ???

# TODO: HumanInTheLoopMiddleware を設定してください
# - send_email と delete_record の実行前に interrupt が発生するようにする
# hitl_middleware = ???

# TODO: HITL エージェントを作成してください
# hitl_agent = create_agent(
#     model="openai:gpt-4o",
#     tools=[send_email, delete_record, get_user_info],
#     middleware=???,     # HumanInTheLoopMiddleware を渡す
#     checkpointer=???,   # InMemorySaver を渡す（HITL に必須）
#     system_prompt=???   # 適切なシステムプロンプトを記述
# )

# print("HITL エージェントの作成が完了しました")

## 課題2: Interrupt（中断）の発生を確認する

**目標:** エージェントにメール送信を依頼し、interrupt が発生することを確認してください。

**ヒント:**
- `agent.invoke()` に `version="v2"` を渡すことで interrupt 情報が取得できます
- 結果は `result.interrupts` で確認できます

In [ ]:
# TODO: thread_id を含む config を定義してください
# config = ???

# TODO: エージェントに以下を依頼してください:
# 「ユーザーID U999 の情報を取得し、そのメールアドレスに件名『テスト』でメールを送ってください」
# result = hitl_agent.invoke(
#     ???,
#     config=config,
#     version="v2",   # interrupt 情報を取得するために必要
# )

# TODO: interrupt が発生しているか確認してください
# if result.interrupts:
#     print("⚠️ 承認待ちです")
#     for interrupt in result.interrupts:
#         print(f"  ツール: {interrupt['tool_call']['name']}")
#         print(f"  引数: {interrupt['tool_call']['args']}")

## 課題3: Approve または Reject して再開する

**目標:** `input()` でユーザー入力を受け取り、
- `"approve"` の場合は `Command(resume=...)` でツールを実行
- `"reject"` の場合はキャンセルしてエージェントに通知する

**ヒント:**
- `from langgraph.types import Command` でインポート
- 承認: `Command(resume={"decisions": [{"type": "approve"}]})`
- 拒否: `Command(resume={"decisions": [{"type": "reject", "reason": "..."}]})`
- `hitl_agent.invoke(Command(...), config=config, version="v2")` で再開

In [ ]:
# TODO: Command をインポートしてください
# from ??? import ???

# ユーザーに入力を求める
user_decision = input("処理を承認しますか？ (approve/reject): ").strip().lower()

if user_decision == "approve":
    print("承認します...")
    # TODO: Command(resume=...) で承認してエージェントを再開してください
    # resume_result = hitl_agent.invoke(
    #     ???,
    #     config=config,
    #     version="v2",
    # )
    # print(resume_result.messages[-1].content)

elif user_decision == "reject":
    print("拒否します...")
    # TODO: Command(resume=...) で拒否してエージェントに通知してください
    # reject_result = hitl_agent.invoke(
    #     ???,
    #     config=config,
    #     version="v2",
    # )
    # print(reject_result.messages[-1].content)